# RACE Dataset — Preprocessing Pipeline
**Course:** Artificial Intelligence — BS(CS) Spring 2026  
**Institution:** NUCES FAST, Islamabad  
**Notebook:** 1 of 5 — Data Loading, Preprocessing, Feature Engineering, Train/Val/Test Split Management

---
### What this notebook does
1. Loads the RACE dataset from Kaggle input
2. Performs exploratory cleaning and text normalization
3. Engineers lexical hand-crafted features
4. Builds **One-Hot Encoding** (primary) and optional TF-IDF feature matrices
5. Computes cosine-similarity features between article, question, and options
6. Saves processed splits to `/kaggle/working/` for downstream model notebooks

## 0. Install / Import

In [1]:
# ── Standard installs available on Kaggle by default ──────────────────────────
import os, re, string, pickle, warnings
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import scipy.sparse as sp
import joblib

warnings.filterwarnings('ignore')
np.random.seed(42)

# Output directory
OUT = '/kaggle/working/processed'
os.makedirs(OUT, exist_ok=True)
print('Output dir:', OUT)

Output dir: /kaggle/working/processed


## 1. Load RACE Dataset

In [2]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

BASE_PATH = '/kaggle/input/datasets/ankitdhiman7/race-dataset'

def load_split(split_name):
    path = os.path.join(BASE_PATH, f'{split_name}.csv')
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")
    df = pd.read_csv(path)
    print(f"  [{split_name}] initial shape: {df.shape}")
    return df

print("Loading RACE data ...")
# Load only the train dataset to avoid the duplicate files
full_df = load_split('train')

# ── Splitting Logic (70% train, 10% val, 20% test) ────────────────────────────

# Step 1: Split off 20% for the test set. 
# temp_df gets the remaining 80%
temp_df, test_df = train_test_split(full_df, test_size=0.20, random_state=42)

# Step 2: Split the 80% temp_df into train (70%) and val (10%).
# To get 10% of the ORIGINAL total from this 80% chunk, we use test_size = 10/80 = 0.125
train_df, val_df = train_test_split(temp_df, test_size=0.125, random_state=42)

# Re-assign the 'split' column tags so downstream code knows which is which
train_df['split'] = 'train'
val_df['split']   = 'val'
test_df['split']  = 'test'

print("\nNew Custom Splits:")
print(f"  [train] shape: {train_df.shape} ({len(train_df)/len(full_df):.0%})")
print(f"  [val]   shape: {val_df.shape}   ({len(val_df)/len(full_df):.0%})")
print(f"  [test]  shape: {test_df.shape}  ({len(test_df)/len(full_df):.0%})")

# ── Quick schema check ────────────────────────────────────────────────────────
REQUIRED_COLS = ['id', 'article', 'question', 'A', 'B', 'C', 'D', 'answer']
assert all(c in train_df.columns for c in REQUIRED_COLS), \
    f"Missing columns. Found: {train_df.columns.tolist()}"
print("\nColumn schema OK:", train_df.columns.tolist())

Loading RACE data ...
  [train] initial shape: (87866, 9)

New Custom Splits:
  [train] shape: (61505, 10) (70%)
  [val]   shape: (8787, 10)   (10%)
  [test]  shape: (17574, 10)  (20%)

Column schema OK: ['Unnamed: 0', 'id', 'article', 'question', 'A', 'B', 'C', 'D', 'answer', 'split']


In [3]:
# Quick peek
train_df.head(2)

,Unnamed: 0,id,article,question,A,B,C,D,answer,split
35413,35413,high6094.txt,Research shows that humans switch from selfish...,What is the text mainly about?,It describes changed behavior when observed.,It details ways to control people's behavior.,It tells how to make people work harder.,It discusses different advertising methods.,A,train
85872,85872,high2797.txt,Doctors say anger can be an extremely damaging...,Expressing anger ferociously _ repressing...,is just the same as,is more harmful than,is no better than,is much better than,C,train


## 2. Basic Quality Checks & Missing-Value Audit

In [4]:
for name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    nulls = df[REQUIRED_COLS].isnull().sum()
    print(f"\n=== {name} null counts ===")
    print(nulls[nulls > 0] if nulls.any() else "  No nulls — clean!")

# Validate answer labels
valid_labels = {'A', 'B', 'C', 'D'}
for name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    bad = df[~df['answer'].isin(valid_labels)]
    print(f"{name}: {len(bad)} rows with invalid answer labels")


=== train null counts ===
A    2
D    6
dtype: int64

=== val null counts ===
A    1
C    1
D    3
dtype: int64

=== test null counts ===
A    1
dtype: int64
train: 0 rows with invalid answer labels
val: 0 rows with invalid answer labels
test: 0 rows with invalid answer labels


## 3. Text Normalization

In [5]:
def normalize_text(text: str) -> str:
    """Lowercase, strip punctuation, collapse whitespace."""
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'[^\w\s]', ' ', text)   # remove punctuation
    text = re.sub(r'\s+', ' ', text).strip()
    return text

TEXT_COLS = ['article', 'question', 'A', 'B', 'C', 'D']

for df in [train_df, val_df, test_df]:
    for col in TEXT_COLS:
        df[col + '_clean'] = df[col].apply(normalize_text)

# Drop rows with empty article or question after cleaning
for name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    before = len(df)
    df.drop(df[(df['article_clean'].str.len() < 10) |
               (df['question_clean'].str.len() < 3)].index, inplace=True)
    df.reset_index(drop=True, inplace=True)
    print(f"{name}: dropped {before - len(df)} rows after cleaning, {len(df)} remain")

print("\nSample cleaned article snippet:")
print(train_df['article_clean'].iloc[0][:200])

train: dropped 21 rows after cleaning, 61484 remain
val: dropped 3 rows after cleaning, 8784 remain
test: dropped 3 rows after cleaning, 17571 remain

Sample cleaned article snippet:
research shows that humans switch from selfish to unselfish behavior when they are watched do you a picture of a set of eyes on a computer screen can cause a change in the way people act even images o


## 4. Combine Train + Val + Test (full corpus for vocabulary building)

In [6]:
all_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
print(f"Full corpus size: {len(all_df):,} rows")

# Build a combined text column for vocabulary / OHE purposes
all_df['combined_text'] = (
    all_df['article_clean'] + ' ' +
    all_df['question_clean'] + ' ' +
    all_df['A_clean'] + ' ' +
    all_df['B_clean'] + ' ' +
    all_df['C_clean'] + ' ' +
    all_df['D_clean']
)
print("Combined text column built.")

Full corpus size: 87,839 rows
Combined text column built.


## 5. Handcrafted Lexical Features

In [7]:
# ── Stop-word list (minimal, no NLTK needed) ───────────────────────────────────
STOP_WORDS = set("a an the is was are were be been being have has had do does did "
                 "will would could should may might shall can must of in on at to "
                 "for with by from up about into through during before after above "
                 "below between out off over under again further then once here "
                 "there when where why how all both each few more most other some "
                 "such no nor not only own same so than too very just i me my "
                 "myself we our ours ourselves you your yours yourself yourselves "
                 "he him his himself she her hers herself it its itself they them "
                 "their theirs themselves what which who whom this that these those "
                 "am and but or as if while because as until although since "
                 "although though even also".split())

def content_words(text: str) -> set:
    return {w for w in text.split() if w not in STOP_WORDS and len(w) > 2}

def word_overlap_ratio(set_a: set, set_b: set) -> float:
    """Jaccard-style overlap ratio."""
    if not set_a or not set_b:
        return 0.0
    return len(set_a & set_b) / len(set_a | set_b)

def char_len(text) -> int:
    return len(str(text)) if isinstance(text, str) else 0

def word_len(text) -> int:
    return len(str(text).split()) if isinstance(text, str) else 0

def build_lexical_features(df: pd.DataFrame) -> pd.DataFrame:
    feats = pd.DataFrame(index=df.index)

    art_words = df['article_clean'].apply(content_words)
    q_words   = df['question_clean'].apply(content_words)

    for opt in ['A', 'B', 'C', 'D']:
        opt_words = df[f'{opt}_clean'].apply(content_words)
        feats[f'overlap_art_{opt}'] = [word_overlap_ratio(a, o)
                                        for a, o in zip(art_words, opt_words)]
        feats[f'overlap_q_{opt}']   = [word_overlap_ratio(q, o)
                                        for q, o in zip(q_words, opt_words)]
        feats[f'len_char_{opt}']    = df[f'{opt}_clean'].apply(char_len)
        feats[f'len_word_{opt}']    = df[f'{opt}_clean'].apply(word_len)

    feats['article_len_words']  = df['article_clean'].apply(word_len)
    feats['article_len_chars']  = df['article_clean'].apply(char_len)
    feats['question_len_words'] = df['question_clean'].apply(word_len)

    # Wh-word flags in question
    for wh in ['who', 'what', 'where', 'when', 'why', 'how', 'which']:
        feats[f'q_has_{wh}'] = df['question_clean'].str.contains(
            rf'\b{wh}\b', regex=True).astype(int)

    # Blank-fill flag
    feats['q_is_blank_fill'] = df['question_clean'].str.contains(
        r'_+', regex=True).astype(int)

    return feats

print("Building lexical features ...")
lex_train = build_lexical_features(train_df)
lex_val   = build_lexical_features(val_df)
lex_test  = build_lexical_features(test_df)

print(f"Lexical feature shape (train): {lex_train.shape}")
lex_train.head(3)

Building lexical features ...
Lexical feature shape (train): (61484, 27)


,overlap_art_A,overlap_q_A,len_char_A,len_word_A,overlap_art_B,overlap_q_B,len_char_B,len_word_B,overlap_art_C,overlap_q_C,...,article_len_chars,question_len_words,q_has_who,q_has_what,q_has_where,q_has_when,q_has_why,q_has_how,q_has_which,q_is_blank_fill
0,0.009174,0.0,43,6,0.018349,0.0,44,8,0.018349,0.000000,...,1645,6,0,1,0,0,0,0,0,0
1,0.000000,0.0,19,5,0.000000,0.0,20,4,0.000000,0.000000,...,1593,10,0,0,0,0,0,0,0,1
2,0.016000,0.0,44,7,0.016000,0.0,37,7,0.007937,0.142857,...,1630,6,0,0,0,0,1,0,0,0


## 6. Label Encoding (Answer → 0/1/2/3)

In [8]:
le = LabelEncoder()
le.fit(['A', 'B', 'C', 'D'])

for df in [train_df, val_df, test_df]:
    df['label'] = le.transform(df['answer'])

print("Label distribution (train):")
print(train_df['label'].value_counts().sort_index())
joblib.dump(le, f'{OUT}/label_encoder.pkl')
print("Label encoder saved.")

Label distribution (train):
label
0    13308
1    15892
2    16743
3    15541
Name: count, dtype: int64
Label encoder saved.


## 7. Build One-Hot Encoding (OHE) Vocabulary

**Primary feature representation** as specified in the project spec.  
We build a vocabulary from the training corpus only (to avoid data leakage),  
then encode `article + question + chosen_option` as a binary bag-of-words vector.

> **Why OHE here?** The project spec requires One-Hot Encoding as the primary  
> classical ML feature representation. We use `CountVectorizer(binary=True)`  
> which is exactly a vocabulary-based one-hot bag-of-words.

In [9]:
# Create combined_text for train_df specifically
train_df['combined_text'] = (
    train_df['article_clean'] + ' ' +
    train_df['question_clean'] + ' ' +
    train_df['A_clean'] + ' ' +
    train_df['B_clean'] + ' ' +
    train_df['C_clean'] + ' ' +
    train_df['D_clean']
)

# Build vocabulary on training corpus combined text only
MAX_VOCAB = 20_000   # cap to keep memory manageable on Kaggle

ohe_vec = CountVectorizer(
    binary=True,
    max_features=MAX_VOCAB,
    min_df=3,          # ignore very rare tokens
    stop_words=None    # we already cleaned
)

ohe_vec.fit(train_df['combined_text'])
print(f"OHE vocabulary size: {len(ohe_vec.vocabulary_):,}")
joblib.dump(ohe_vec, f'{OUT}/ohe_vectorizer.pkl')

OHE vocabulary size: 20,000


['/kaggle/working/processed/ohe_vectorizer.pkl']

In [10]:
# ── Per-option feature matrices ────────────────────────────────────────────────
# For answer verification: concatenate [article_clean + question_clean + option_clean]
# for EACH option. This gives 4x rows per original sample.

def build_ohe_verification_matrix(df: pd.DataFrame, vec):
    """
    Returns:
        X_ohe  : sparse matrix, shape (N*4, vocab)
        y      : array of 0/1 correct labels, shape (N*4,)
        meta   : DataFrame with original index, option, answer
    """
    texts, labels, metas = [], [], []
    for _, row in df.iterrows():
        for opt in ['A', 'B', 'C', 'D']:
            combined = (row['article_clean'] + ' ' +
                        row['question_clean'] + ' ' +
                        row[f'{opt}_clean'])
            texts.append(combined)
            labels.append(1 if row['answer'] == opt else 0)
            metas.append({'orig_index': row.name, 'option': opt,
                          'answer': row['answer']})
    X = vec.transform(texts)
    y = np.array(labels)
    return X, y, pd.DataFrame(metas)

print("Encoding train set (this may take ~1-3 min) ...")
X_train_ohe, y_train_ohe, meta_train = build_ohe_verification_matrix(train_df, ohe_vec)
print(f"  X_train_ohe shape : {X_train_ohe.shape}")

print("Encoding val set ...")
X_val_ohe, y_val_ohe, meta_val = build_ohe_verification_matrix(val_df, ohe_vec)
print(f"  X_val_ohe shape   : {X_val_ohe.shape}")

print("Encoding test set ...")
X_test_ohe, y_test_ohe, meta_test = build_ohe_verification_matrix(test_df, ohe_vec)
print(f"  X_test_ohe shape  : {X_test_ohe.shape}")

Encoding train set (this may take ~1-3 min) ...
  X_train_ohe shape : (245936, 20000)
Encoding val set ...
  X_val_ohe shape   : (35136, 20000)
Encoding test set ...
  X_test_ohe shape  : (70284, 20000)


In [11]:
# Save sparse matrices
sp.save_npz(f'{OUT}/X_train_ohe.npz', X_train_ohe)
sp.save_npz(f'{OUT}/X_val_ohe.npz',   X_val_ohe)
sp.save_npz(f'{OUT}/X_test_ohe.npz',  X_test_ohe)
np.save(f'{OUT}/y_train_ohe.npy', y_train_ohe)
np.save(f'{OUT}/y_val_ohe.npy',   y_val_ohe)
np.save(f'{OUT}/y_test_ohe.npy',  y_test_ohe)
meta_train.to_csv(f'{OUT}/meta_train.csv', index=False)
meta_val.to_csv(f'{OUT}/meta_val.csv',     index=False)
meta_test.to_csv(f'{OUT}/meta_test.csv',   index=False)
print("OHE matrices saved.")

OHE matrices saved.


## 8. (Optional) TF-IDF Feature Matrix

TF-IDF is optional per the spec. We build it here for completeness;  
downstream models may choose to load either OHE or TF-IDF.

In [12]:
tfidf_vec = TfidfVectorizer(
    max_features=MAX_VOCAB,
    min_df=3,
    sublinear_tf=True
)
tfidf_vec.fit(train_df['combined_text'])
joblib.dump(tfidf_vec, f'{OUT}/tfidf_vectorizer.pkl')

def build_tfidf_matrix(df, vec):
    texts = []
    for _, row in df.iterrows():
        for opt in ['A', 'B', 'C', 'D']:
            combined = (row['article_clean'] + ' ' +
                        row['question_clean'] + ' ' +
                        row[f'{opt}_clean'])
            texts.append(combined)
    return vec.transform(texts)

print("Building TF-IDF matrices ...")
X_train_tfidf = build_tfidf_matrix(train_df, tfidf_vec)
X_val_tfidf   = build_tfidf_matrix(val_df,   tfidf_vec)
X_test_tfidf  = build_tfidf_matrix(test_df,  tfidf_vec)

sp.save_npz(f'{OUT}/X_train_tfidf.npz', X_train_tfidf)
sp.save_npz(f'{OUT}/X_val_tfidf.npz',   X_val_tfidf)
sp.save_npz(f'{OUT}/X_test_tfidf.npz',  X_test_tfidf)
print(f"TF-IDF train shape: {X_train_tfidf.shape} — saved.")

Building TF-IDF matrices ...
TF-IDF train shape: (245936, 20000) — saved.


## 9. Cosine Similarity Features

In [13]:
from sklearn.metrics.pairwise import paired_cosine_distances

def build_cosine_features(df: pd.DataFrame, vec) -> pd.DataFrame:
    """
    Vectorized version of cosine similarity feature generation.
    Computes all row-wise similarities instantly without slow for-loops.
    """
    # 1. Transform text into sparse matrices
    art_vecs = vec.transform(df['article_clean'])
    q_vecs   = vec.transform(df['question_clean'])
    opt_vecs = {opt: vec.transform(df[f'{opt}_clean']) for opt in ['A', 'B', 'C', 'D']}

    feats = {}
    
    # 2. Compute paired cosine similarities (1 - distance) for all rows at once
    for opt in ['A', 'B', 'C', 'D']:
        feats[f'cos_art_{opt}'] = 1 - paired_cosine_distances(art_vecs, opt_vecs[opt])
        feats[f'cos_q_{opt}']   = 1 - paired_cosine_distances(q_vecs, opt_vecs[opt])
        
    feats['cos_art_q'] = 1 - paired_cosine_distances(art_vecs, q_vecs)
    
    # 3. Format into DataFrame
    cols = ([f'cos_art_{o}' for o in 'ABCD'] +
            [f'cos_q_{o}'   for o in 'ABCD'] +
            ['cos_art_q'])
            
    return pd.DataFrame(feats, columns=cols, index=df.index)

print("Computing cosine features (train) ...")
cos_train = build_cosine_features(train_df, ohe_vec)

print("Computing cosine features (val) ...")
cos_val   = build_cosine_features(val_df, ohe_vec)

print("Computing cosine features (test) ...")
cos_test  = build_cosine_features(test_df, ohe_vec)

cos_train.to_csv(f'{OUT}/cos_features_train.csv', index=False)
cos_val.to_csv(f'{OUT}/cos_features_val.csv',     index=False)
cos_test.to_csv(f'{OUT}/cos_features_test.csv',   index=False)
print("Cosine features saved.")

Computing cosine features (train) ...
Computing cosine features (val) ...
Computing cosine features (test) ...
Cosine features saved.


## 10. Save Cleaned DataFrames & Lexical Features

In [14]:
# Merge lexical features into cleaned DataFrames
train_full = pd.concat([train_df.reset_index(drop=True),
                        lex_train.reset_index(drop=True),
                        cos_train.reset_index(drop=True)], axis=1)
val_full   = pd.concat([val_df.reset_index(drop=True),
                        lex_val.reset_index(drop=True),
                        cos_val.reset_index(drop=True)], axis=1)
test_full  = pd.concat([test_df.reset_index(drop=True),
                        lex_test.reset_index(drop=True),
                        cos_test.reset_index(drop=True)], axis=1)

train_full.to_csv(f'{OUT}/train_processed.csv', index=False)
val_full.to_csv(f'{OUT}/val_processed.csv',     index=False)
test_full.to_csv(f'{OUT}/test_processed.csv',   index=False)

print("All processed files saved to:", OUT)
print("train_processed shape:", train_full.shape)
print("val_processed shape:  ", val_full.shape)
print("test_processed shape: ", test_full.shape)

All processed files saved to: /kaggle/working/processed
train_processed shape: (61484, 54)
val_processed shape:   (8784, 53)
test_processed shape:  (17571, 53)


## 11. Summary Checklist

In [15]:
import os
files = os.listdir(OUT)
files.sort()
print(f"Files in {OUT}:")
for f in files:
    size = os.path.getsize(os.path.join(OUT, f))
    print(f"  {f:40s}  {size/1e6:.2f} MB")

print("\n✅ Preprocessing complete. Proceed to notebook 2 (model_a_train) or 4 (EDA).")

Files in /kaggle/working/processed:
  X_test_ohe.npz                            5.85 MB
  X_test_tfidf.npz                          84.96 MB
  X_train_ohe.npz                           20.35 MB
  X_train_tfidf.npz                         295.73 MB
  X_val_ohe.npz                             2.92 MB
  X_val_tfidf.npz                           42.32 MB
  cos_features_test.csv                     2.96 MB
  cos_features_train.csv                    10.36 MB
  cos_features_val.csv                      1.48 MB
  label_encoder.pkl                         0.00 MB
  meta_test.csv                             0.66 MB
  meta_train.csv                            2.41 MB
  meta_val.csv                              0.31 MB
  ohe_vectorizer.pkl                        0.58 MB
  test_processed.csv                        66.60 MB
  tfidf_vectorizer.pkl                      0.74 MB
  train_processed.csv                       336.39 MB
  val_processed.csv                         33.22 MB
  y_test_ohe.npy  

In [16]:
import shutil
from IPython.display import FileLink

# Create the zip file
shutil.make_archive(
    base_name='/kaggle/working/processed_data', 
    format='zip', 
    root_dir='/kaggle/working/processed'
)

print("Zip file created successfully! Click the link below to download:")

# Display a clickable link to the generated zip file
# (Kaggle expects relative paths for FileLink)
display(FileLink('processed_data.zip'))

Zip file created successfully! Click the link below to download:


/kaggle/working/processed_data.zip